# 03 — Single-Position Deep Dive

**Notebook 3 of the *Developer Guide to Disciplined Trading* series.**

> Prerequisites: [`01-foundations`](./01-foundations-techtrade-and-analysis.ipynb), [`02-morning-scan`](./02-morning-scan.ipynb). You should already have run a scan and have a candidate ticker in mind.

---

## Learning objectives

By working through this notebook you'll:

1. Understand how a raw [**confluence signal**](https://www.investopedia.com/terms/c/confluence.asp) becomes a fully-specified [**trade plan**](https://www.investopedia.com/terms/t/trading-plan.asp) — entry, stop, target, sizing, and monitoring rules.
2. See how [**ATR-based stops**](https://www.investopedia.com/articles/trading/08/atr.asp) adapt position sizing to each stock's own volatility (the [**volatility-normalized sizing**](https://www.investopedia.com/terms/v/volatility.asp) technique the Turtle Traders made famous).
3. Learn what [**order legs**](https://www.investopedia.com/terms/o/order.asp) are and why professional trading systems split a single trade into multiple pre-registered orders ([entry](https://www.investopedia.com/terms/e/entryorder.asp) + [stop-loss](https://www.investopedia.com/terms/s/stop-lossorder.asp) + [take-profit](https://www.investopedia.com/terms/t/take-profitorder.asp) + time exit).
4. Grasp the [**no-look-ahead bias**](https://www.investopedia.com/terms/l/lookaheadbias.asp) discipline — why bar-`t` signals must fill at bar `t+1`, and why this is the #1 way retail backtests silently lie to their authors.
5. See [**slippage**](https://www.investopedia.com/terms/s/slippage.asp) and [**commission**](https://www.investopedia.com/terms/c/commission.asp) modeled as real trade costs — the two frictions that turn a "profitable in backtest" system into a losing one in production.

## Alex's question this notebook answers

> *"The morning scan flagged NVDA. Should I take the trade — and if I do, what does the engine think the position should look like at the bar-by-bar level?"*

Notebook 02 produced a ranked list across all 11 sectors. **This notebook zooms in on ONE name** and shows the full single-position pipeline that produces it:

1. **`signals`** — the per-symbol confluence score + the indicator vote attribution (read "why this score?")
2. **`plan`** — turn the signal into a complete `TradePlan` with entry / stop / target levels, position size, and an inline `Recommendation`
3. **`orders`** — materialize the broker-ready order legs (entry + stop + target + time-exit)
4. **`simulate`** — paper-fill those order legs against a forward bar window. **No look-ahead** — bar-`t` signals only fill at `t+1`.

## Why zoom in on one name?

The morning scan is a **funnel** — it takes ~200 candidates across 11 sectors, filters to ~5-15 High-conviction plans, and dumps them into Excel. But the funnel treats every candidate uniformly; **the deep-dive is where a human interrogates the pick**. Two reasons this matters:

1. **Position sizing is per-position, not per-portfolio.** The scan's `qty` is a *default*; the deep-dive is where you decide whether this specific setup deserves default size, half size, or skip. See [Investopedia: Position Sizing](https://www.investopedia.com/terms/p/positionsizing.asp).
2. **Every professional trading desk has a "second-look" step.** Sell-side analysts pitch; PMs interrogate. Quant funds' models rank; risk managers veto. The deep-dive is Alex's version of the [**four-eyes principle**](https://en.wikipedia.org/wiki/Two-person_rule) — a discipline that catches the ~5% of scan flags that shouldn't have been flagged.

## What Alex takes away

By the end he can answer:

- **What does each order leg do?** (entry vs exit_stop vs exit_target vs exit_time)
- **Where would today's signal have actually filled?** (next-bar-open with slippage + commission)
- **Which exit triggered first** on the simulated path (the answer is *not always* the one Alex would have guessed).
- **How much would a wider stop have cost in position size** (the risk-budget-vs-stop-distance trade-off, made concrete).

## Wall-clock

~3-5 min warm. Most of it is the OHLCV fetch for the chosen symbol; the engine math is sub-second.

## What this notebook is NOT

- **Not a backtest.** A single forward-window simulation is one sample; it's an anecdote, not statistical evidence. Notebook 04 does the actual validation.
- **Not permission to trade.** The final decision is Alex's; the engine's job is to enforce the discipline of the pre-trade checklist, not to override it. Read the SEC's [investor.gov beginner guidance](https://www.investor.gov/introduction-investing/investing-basics/save-and-invest) before deploying real capital.

## Choose your ticker

We use `NVDA` as the running example because it's [**liquid**](https://www.investopedia.com/terms/l/liquidity.asp), well-known, and almost always has a non-flat signal. **Swap `SYMBOL` below** for any ticker the scan flagged for you. If you don't have one yet, run notebook 02 first.

---

*Reading time: ~25 minutes with the linked concepts. Coding time: 5 minutes for the whole notebook.*

In [ ]:
# Edit this and re-run the rest of the notebook.
SYMBOL = "NVDA"
PRESET = "trend_follow"  # try "mean_revert" or "breakout" for the same ticker — surprising differences
RISK = 0.01              # 1% of notional per trade

## 1. Setup probe

Same fail-fast smoke-test as notebooks 01 and 02: cheap sanity check before any expensive call. If this cell errors, fix the environment before proceeding — the rest of the notebook assumes `obb` and `fmp_cached` are both healthy.

In [ ]:
from openbb import obb
from datetime import date
print(f"obb loaded; today is {date.today().isoformat()}")
print(f"Studying {SYMBOL} under preset='{PRESET}' at risk={RISK:.1%}")

## 2. The raw signal — `obb.techtrade.signals`

Before any plan or order assembly, just compute the confluence score for this one ticker. `signals` is the lightest call in the pipeline: it builds the indicator panel, runs the confluence engine under the chosen preset, and returns the score + the vote attribution. No sizing, no orders.

### What is a "signal" formally?

In systematic trading, a [**signal**](https://www.investopedia.com/terms/s/trading-signal.asp) is a numerical output on `[-1, +1]` (or similar bounded scale) that answers **one question**: *given today's data, what direction and how strongly should we trade this symbol?* Positive = long-biased; negative = short-biased; magnitude = conviction. The signal is deliberately kept separate from sizing and order construction (see §3 and §4) because the same signal can drive different trades depending on account size, risk tolerance, and broker execution constraints — [**separation of concerns**](https://en.wikipedia.org/wiki/Separation_of_concerns) applied to trading system design.

### Why "confluence" instead of a single indicator?

Any single indicator — RSI, MACD, moving-average crossovers, whatever — is right ~55% of the time under favorable conditions and ~45% under adverse ones (see David Aronson's *[Evidence-Based Technical Analysis](https://www.wiley.com/en-us/Evidence+Based+Technical+Analysis%3A+Applying+the+Scientific+Method+and+Statistical+Inference+to+Trading+Signals-p-9780470008744)*, Wiley 2007, for statistical evidence). No single indicator gives you an edge worth trading on retail-size capital after slippage and commissions.

But **independent noisy signals combined intelligently** — the [**ensemble method**](https://en.wikipedia.org/wiki/Ensemble_learning) from ML — can produce a composite whose accuracy exceeds any individual component. This is what a [**confluence signal**](https://www.investopedia.com/terms/c/confluence.asp) does. Marcos López de Prado's [*Advances in Financial Machine Learning*](https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086) (Wiley 2018) is the modern reference for why ensemble techniques dominate systematic trading; the trend/momentum/volatility/volume panel used here is a domain-specific ensemble.

**Wall-clock:** ~3-8 seconds (one OHLCV fetch + indicator math).

In [ ]:
sigs = obb.techtrade.signals(symbols=[SYMBOL], preset=PRESET).results
if not sigs:
    raise RuntimeError(f"No signal for {SYMBOL} — maybe the symbol is unsupported on fmp_cached.")
sig = sigs[0]
print(f"{sig.symbol}  as_of={sig.as_of}  segment={sig.segment}")
print(f"  composite score:  {sig.score:+.4f}  direction: {sig.direction}")
print(f"  vote count:       {len(sig.votes)}")

### Read the votes

Every indicator that contributed to the score is shown with its `family / name / vote / weight`. The composite is the weighted sum (with the volume family acting as a multiplier, not an additive term — see PRD §12.2). The cell below sorts by absolute contribution so the loudest voters are at the top.

### The 5 attribution columns — what each one means

| Column | Type | Range | What it answers | Deeper reading |
|---|---|---|---|---|
| **`family`** | categorical | one of 4 | *Which "category of evidence" is this indicator?* Techtrade groups indicators into 4 families that answer different questions about price action. See table below. | [Investopedia: Technical Indicator](https://www.investopedia.com/terms/t/technicalindicator.asp) |
| **`name`** | string | e.g. `sma_50_200_cross`, `rsi_14`, `atr_pct` | *Which specific indicator?* The named formula that produced the vote — click through to the family row to see the classical reference. | [Investopedia: Technical Analysis](https://www.investopedia.com/terms/t/technicalanalysis.asp) |
| **`vote`** | float | `[-1.0, +1.0]` | *In which direction and how strongly does this indicator think?* `+1` = maximally bullish; `-1` = maximally bearish; `0` = neutral / no signal. Normalized so different indicators are directly comparable. | [Investopedia: Signal](https://www.investopedia.com/terms/s/trading-signal.asp) |
| **`weight`** | float | `[0.0, 1.0]` | *How much should we listen to this indicator's family?* Set by the preset; the four families' weights sum to 1.0. E.g. `trend_follow` gives trend 0.40, momentum 0.25, volatility 0.20, volume 0.15. See [ensemble weighting](https://en.wikipedia.org/wiki/Ensemble_learning). | [Investopedia: Weighted Average](https://www.investopedia.com/terms/w/weightedaverage.asp) |
| **`contrib`** | float | `[-weight, +weight]` | *This indicator's signed contribution to the composite score.* Computed as `vote × weight`. Sorting by `abs(contrib)` shows the loudest voices at the top — those are the indicators actually driving the recommendation. | [Investopedia: Contribution Analysis](https://www.investopedia.com/terms/c/contribution-analysis.asp) |

The composite score is (approximately) `sum(contrib)` across all rows — with the important nuance that the [**volume**](https://www.investopedia.com/terms/v/volume.asp) family is applied as a **multiplier**, not summed, so it amplifies or damps the trend/momentum/volatility trio rather than adding an independent vote.

### The 4 indicator families — full reference

| Family | Weight (`trend_follow`) | Question it answers | Example indicators (with Investopedia links) |
|---|---:|---|---|
| **[Trend](https://www.investopedia.com/terms/t/trendtrading.asp)** | **0.40** | *Which direction is price going, and how strongly?* | [SMA / EMA crossovers](https://www.investopedia.com/terms/s/sma.asp), [ADX](https://www.investopedia.com/terms/a/adx.asp), [Ichimoku](https://www.investopedia.com/terms/i/ichimoku-cloud.asp), [Aroon](https://www.investopedia.com/terms/a/aroon.asp) |
| **[Momentum](https://www.investopedia.com/terms/m/momentum.asp)** | **0.25** | *How fast is price moving? Overbought or oversold?* | [RSI](https://www.investopedia.com/terms/r/rsi.asp), [MACD](https://www.investopedia.com/terms/m/macd.asp), [Stochastic Oscillator](https://www.investopedia.com/terms/s/stochasticoscillator.asp), [ROC](https://www.investopedia.com/terms/p/pricerateofchange.asp) |
| **[Volatility](https://www.investopedia.com/terms/v/volatility.asp)** | **0.20** | *What range should we expect? Is vol compressing or expanding?* | [ATR](https://www.investopedia.com/terms/a/atr.asp), [Bollinger Bands](https://www.investopedia.com/terms/b/bollingerbands.asp), [Keltner Channels](https://www.investopedia.com/terms/k/keltnerchannel.asp), [Historical Volatility](https://www.investopedia.com/terms/h/historicalvolatility.asp) |
| **[Volume](https://www.investopedia.com/terms/v/volume.asp)** | **0.15** ⚡ multiplier | *Is real money behind the move, or is it thin?* | [OBV](https://www.investopedia.com/terms/o/onbalancevolume.asp), [Chaikin Money Flow](https://www.investopedia.com/terms/c/chaikinmoneyflow.asp), [MFI](https://www.investopedia.com/terms/m/mfi.asp), [VWAP](https://www.investopedia.com/terms/v/vwap.asp), [Accumulation/Distribution](https://www.investopedia.com/terms/a/accumulationdistribution.asp) |

**Preset differences at a glance:**

| Preset | Trend | Momentum | Volatility | Volume |
|---|---:|---:|---:|---:|
| `trend_follow` (default) | **0.40** | 0.25 | 0.20 | 0.15 |
| `mean_revert` | 0.15 | **0.40** | 0.30 | 0.15 |
| `breakout` | 0.25 | 0.15 | **0.35** | **0.25** |

Try re-running §7 with a different preset and watch the same `vote` values produce very different `contrib` — that's the preset table swapping which families "get heard" loudest.

### Why "attribution" matters

A confluence score without attribution is just a number. **Attribution** — the vote-by-vote breakdown — is what separates a professional-grade signal from a black-box "buy this" alert. Two reasons attribution is worth paying for:

1. **Debuggability.** If a plan loses money, attribution tells you *which family failed*. A losing trend-follow signal where the trend family voted correctly but momentum whipsawed is a different failure mode than one where trend itself was wrong. Different failure modes call for different fixes (or acceptance of variance) — you cannot distinguish them without attribution. This is [**model diagnostics**](https://en.wikipedia.org/wiki/Regression_diagnostic) borrowed from statistics; the finance-specific term is [**performance attribution**](https://www.investopedia.com/terms/a/attribution-analysis.asp).
2. **Psychology.** When Alex is tempted to override the engine (*"I have a feeling"*), attribution is the counterargument: *"seven independent indicators voted long with weight X, my feeling is one data point."* The [SEC's investor education](https://www.investor.gov/introduction-investing/investing-basics/how-stock-markets-work/how-market-works) is unambiguous that emotional overrides are the #1 destroyer of retail portfolio value. Attribution is your defense.

The bar to clear for a "clean" signal (see §2 interpretation cell below): **at least 3 out of 4 families vote the same direction**, and no single high-weight family votes the opposite direction. Anything less is a mixed signal — technically it might exceed the entry threshold, but the underlying evidence is contradictory.

In [ ]:
import pandas as pd

votes_df = pd.DataFrame([
    {"family": v.family, "name": v.name, "vote": v.vote, "weight": v.weight, "contrib": v.vote * v.weight}
    for v in sig.votes
])
votes_df = votes_df.reindex(votes_df["contrib"].abs().sort_values(ascending=False).index).reset_index(drop=True)
votes_df

### What to look for

- **Sign agreement across families.** If [trend](https://www.investopedia.com/terms/t/trendtrading.asp) + [momentum](https://www.investopedia.com/terms/m/momentum.asp) + [volatility](https://www.investopedia.com/terms/v/volatility.asp) all vote `+`, the score is the result of confluence not coincidence. This is the same "multiple independent evidences agreeing" principle that gives [**cross-validation**](https://en.wikipedia.org/wiki/Cross-validation_(statistics)) its statistical power.
- **A high-weight family voting against.** Weight 0.40 on trend; if trend votes `-1.0` while every other family votes `+`, the composite might still cross the threshold but **the strongest voice is dissenting**. That's a setup Alex doesn't take. This is analogous to a court trial where the DNA evidence contradicts the eyewitnesses — you don't average them; you weight the higher-fidelity evidence more.
- **[Volume](https://www.investopedia.com/terms/v/volume.asp).** Volume votes are **amplification, not addition**. Negative volume votes ([distribution](https://www.investopedia.com/terms/d/distribution.asp)) damp the score; positive ([accumulation](https://www.investopedia.com/terms/a/accumulation.asp)) amplify. Why? Because volume answers *"how much real money is behind this move?"* — see [Investopedia on OBV](https://www.investopedia.com/terms/o/onbalancevolume.asp) for the classic Joe Granville formulation and [Chaikin Money Flow](https://www.investopedia.com/terms/c/chaikinmoneyflow.asp) for a modern refinement. A price move on tiny volume is a small number of participants agreeing; on high volume it's institutional consensus.

### The classical reference

If you want to dig deeper into any specific indicator: John Murphy's [*Technical Analysis of the Financial Markets*](https://www.investopedia.com/articles/active-trading/010615/top-technical-analysis-books.asp) (NYIF, 1999) is the industry-standard textbook — reads like a dictionary, gives you the formula, standard periods, and typical use cases for every indicator you'll encounter.

### Note on panel size (bd-7ct foundation + bd-luy trend family, updated 2026-07-10)

The panel this notebook computes by default is the **classic** 14-key / 7-vote confluence panel (5 trend keys + 3 momentum + 4 volatility + 2 volume). An **extended** panel is being rolled out family-by-family, each gated on measured out-of-sample forward Information Coefficient.

**Shipped so far — trend family (bd-luy):**

- **Aroon Up / Down / Oscillator** — Chande 1995, 25-bar canonical. Measures the recency of the 25-bar high vs the 25-bar low. Bounded oscillator in `[-100, +100]`. Vote = `clip(aroon_osc / 100, -1, +1)`. Complements the moving-average / MACD family by capturing "how recent was the last new extreme" rather than "which side of the moving average are we on."
- **Ichimoku Cloud** — Hosoda 1969, 9/26/52 canonical. Compares close to the current-bar leading spans (senkou A/B); +1 above cloud, −1 below, 0 inside. **Vote reads a 3-bar-confirmation buffer** — a 1- or 2-bar whipsaw through cloud does NOT flip the vote (§R.4 M3 hysteresis, symmetric with PSAR treatment). Requires 78+ bars of history (senkou B needs 52 lookback + 26-bar forward displacement); degrades gracefully on shorter series.

**Deliberately cut from bd-luy scope:** Parabolic SAR — Han/Yang/Zhou (2011, *RFS*) and Neely et al. (2014, *Management Science*) both show PSAR has negative alpha on US equities after costs. A follow-up bead (`bd-l4ga`) tracks revisiting either PSAR or better alternatives (Donchian breakout, 200-day SMA slope, cross-sectional 3/6/12-month momentum) after shadow-mode data accumulates.

**Known bd-luy follow-up (`bd-hpxh`):** the bd-8332 decorrelation gate surfaced pairwise `|Spearman ρ|` = 0.839 between `ema_cross` and `ichimoku_cloud` on the 5-year basket — above the §R.4 M4 ceiling of 0.70. Design-level decision pending in that bead (recommended: drop `ichimoku_cloud` from the ship config while keeping the panel key for audit). Until resolved, `ichimoku_cloud` vote still emits but the family PR merge is gated on the decision.

**Coming in future family PRs (each gated on IC):**

- **bd-40v (momentum)** — ROC (10/20) + CCI (20). Cut from original design: Williams %R (exact affine of Stochastic) and MACD signal-line cross (same event as MACD histogram sign).
- **bd-z43 (volatility)** — BB Bandwidth + Keltner channel position. TTM Squeeze + Historical Volatility reclassified as gates/scalers.
- **bd-alj (volume)** — MFI + A/D Line slope + volume ratio. Blocked also by GH #75 (short-side multiplier inversion).

**Reference:**

- Design: [`docs/superpowers/specs/2026-07-08-confluence-panel-expansion-design.md`](../docs/superpowers/specs/2026-07-08-confluence-panel-expansion-design.md) (includes §10 expert review)
- Trading-system validation: [`docs/superpowers/specs/2026-07-10-ensemble-lift-validation-design.md`](../docs/superpowers/specs/2026-07-10-ensemble-lift-validation-design.md) (paired-spread acceptance test + external benchmark panel + point-in-time universe)
- Foundation plan: [`docs/superpowers/plans/2026-07-08-bd-7ct-confluence-foundation.md`](../docs/superpowers/plans/2026-07-08-bd-7ct-confluence-foundation.md)
- Trend family plan: [`docs/superpowers/plans/2026-07-09-bd-luy-trend-family-expansion.md`](../docs/superpowers/plans/2026-07-09-bd-luy-trend-family-expansion.md)

**To opt in to the extended panel** (defaults stay classic — this notebook's tables remain accurate for classic):

```python
AnalysisConfig(
    symbol=SYMBOL,
    feature_flags=AnalysisFeatureFlags(use_extended_confluence_panel=True),
)
```

With flag on, the trend family emits up to 5 votes (classic 3 + `aroon_osc` + `ichimoku_cloud`, subject to the bd-hpxh decision above); other families still emit their classic vote counts pending their family PRs.

## 3. The full plan — `obb.techtrade.plan`

`signals` answered *should I trade?* `plan` answers *if I trade, what's the trade?* The plan adds:

- **Levels**: entry / stop / target prices (sized off [ATR(14)](https://www.investopedia.com/terms/a/atr.asp) and the rule's `atr_stop_mult` + `target_r_multiple`)
- **Sizing**: `position_size` in shares such that the stop-distance × shares ≈ `risk` × notional
- **Orders**: broker-ready order legs with `intent` tags (`entry` / `exit_stop` / `exit_target` / `exit_time` / `exit_signal`)
- **Recommendation**: human-facing summary (`action`, `conviction`, `reasoning`, `caveats`)
- **`validation`**: empty until notebook 04 fills it via `validate`

### The 5 mandatory components of any trade plan

Any trader can invent a trade plan on the back of a napkin. Every professional trade plan has these five components — they map exactly onto what techtrade emits:

| Component | Question it answers | Techtrade field |
|---|---|---|
| **Entry** | At what price do I get in? | `entry_price` |
| **Stop-loss** | At what price do I admit I was wrong and get out? | `stop_price` |
| **Target** | At what price do I take my profit? | `target_price` |
| **Size** | How many shares? | `position_size` |
| **Time stop** | If neither stop nor target hits, when do I close anyway? | `time_stop_bars` |

The [**time stop**](https://www.investopedia.com/terms/t/timedecay.asp) is the component most retail traders forget. Why does it matter? A dead-money position is not free — it's **opportunity cost**. Every dollar tied up in a stock that's going nowhere is a dollar not deployed into a better setup. Ed Seykota's rule of thumb (from Jack Schwager's [*Market Wizards*](https://en.wikipedia.org/wiki/Market_Wizards), NYIF 1989): *"if a trade hasn't proven itself in 5 days, cut it — even if it's not a loser yet."*

### Why compute the stop with ATR instead of a fixed %?

A 2% stop on Coca-Cola and a 2% stop on a small-cap biotech treat two totally different risk profiles the same. [**Average True Range**](https://www.investopedia.com/terms/a/atr.asp) — Welles Wilder's 1978 invention (see his *[New Concepts in Technical Trading Systems](https://www.tradingliteracy.com/best-technical-analysis-books/)*) — measures **each stock's own daily price range**, so a 2× ATR stop is intrinsically wider on volatile stocks and tighter on placid ones.

The consequence: **position size adapts automatically**. A stock with ATR = $5 gets a smaller position than a stock with ATR = $2, because the sizing formula (§below) is `qty = risk_budget / stop_distance`. This is the [**volatility-normalized sizing**](https://en.wikipedia.org/wiki/Volatility_(finance)) technique that Richard Dennis's [**Turtle Traders**](https://en.wikipedia.org/wiki/Turtle_Traders) used to turn $10M into $175M in the mid-1980s — see Michael Covel's [*The Complete TurtleTrader*](https://www.michaelcovel.com/complete-turtletrader/) (HarperCollins 2007) for the historical detail. Same technique appears verbatim in Van Tharp's [*Trade Your Way to Financial Freedom*](https://vantharp.com/trade-your-way-to-financial-freedom/).

### The `Recommendation` object

The `recommendation` field is a **human-facing summary** — action label, conviction bucket, reasoning text, and caveats. It's designed to be readable by a non-programmer over coffee (the Excel export in notebook 02 uses it directly). Under the hood, everything in `recommendation` is deterministically derived from `plan.signal` and the rule config — nothing there requires an LLM, so the same inputs always produce the same recommendation text (reproducibility for post-trade review).

In [ ]:
plans = obb.techtrade.plan(symbols=[SYMBOL], preset=PRESET, risk=RISK).results
if not plans:
    raise RuntimeError(f"plan() returned no plans for {SYMBOL} — score below entry_threshold?")
plan = plans[0]
rec = plan.recommendation

print(f"--- TradePlan for {plan.symbol} ({plan.segment}, as_of {plan.as_of}) ---\n")
print(f"  Action:        {rec.action}  ({rec.conviction})")
print(f"  Score:         {plan.signal.score:+.4f}")
print(f"  Entry price:   ${rec.entry_price}")
print(f"  Stop price:    ${rec.stop_price}  ({rec.stop_distance_pct*100:.2f}% from entry)")
print(f"  Target price:  ${rec.target_price}  ({rec.target_distance_pct*100:.2f}% from entry)")
print(f"  R:R:           {rec.risk_reward:.2f}")
print(f"  ATR(14):       {rec.atr:.2f}")
print(f"  Position size: {rec.position_size} shares")
print(f"  Risk/share:    ${rec.risk_per_share}")
print(f"  Risk % notnl:  {rec.risk_pct_of_notional*100:.3f}%")
print(f"  Time stop:     {rec.time_stop_bars} bars")
print(f"\n  Caveats: {rec.caveats}")

### The sizing math — decoded

The formula the engine uses is:

```
position_size = floor( risk_budget / risk_per_share )
```

where:

- `risk_budget = risk × notional` — the *dollar amount* you're willing to lose if the stop hits. You set `risk` (e.g. 0.01 = 1%); the engine multiplies by your account notional.
- `risk_per_share = entry_price - stop_price` for longs (or `stop - entry` for shorts) — the *dollar loss per share* if the stop fires.

### Why this formula? The invariant it enforces

**Every trade loses exactly `risk_budget` dollars if stopped out.** Not "about" `risk_budget`. Exactly. Because the engine sizes `qty` to make it so.

Concrete example: $100K account, `risk=0.01`, `risk_budget = $1,000`.

| Entry | Stop | Stop distance | qty | Loss if stopped |
|---:|---:|---:|---:|---:|
| $150 | $147.50 | $2.50 | 400 shares | $1,000 |
| $150 | $145.00 | $5.00 | 200 shares | $1,000 |
| $150 | $148.00 | $2.00 | 500 shares | $1,000 |

**Wider stop → smaller position → same dollar loss.** This is the entire point of the [**1% risk rule**](https://www.investopedia.com/articles/trading/09/risk-management.asp) (Van Tharp's central insight): the rule is enforced by *sizing*, not by prayer.

Contrast with the naïve "always buy 100 shares" approach: a $10 wide stop loses $1,000; a $2 wide stop loses $200; a $20 wide stop loses $2,000. Losses vary 10×, and — critically — the biggest losses come from the trades where the market told you it was going to be volatile (wide stop). You'd be maximally exposed to your worst setups. This is [**anti-Kelly sizing**](https://www.investopedia.com/articles/trading/04/091504.asp), and it's why undisciplined traders blow up.

### What "notional" means here

**[Notional value](https://www.investopedia.com/terms/n/notionalvalue.asp)** is the *nominal* dollar value of the position — literally `entry_price × qty`. When Alex says "1% risk on $100K notional" he means: *if my account equity is $100K, I want a stop-out to cost $1,000.* The engine's default `notional` convention is documented in PRD §13; the practical rule of thumb is "use your account equity" for stock trades and "use position size × contract multiplier" for futures.

### Try widening the stop

The cell below shows the trade-off concretely. Watch what happens to `qty` as the ATR multiplier grows: same signal, same entry, wider stop = smaller position — the risk-budget invariant enforcing itself automatically.

In [ ]:
from decimal import Decimal

entry = float(rec.entry_price)
stop = float(rec.stop_price)
atr = rec.atr
qty = float(rec.position_size)

# What if the rule used a tighter or wider ATR multiplier?
# The engine default is atr_stop_mult=2.0 (rule.atr_stop_mult).
print(f"At default 2.0×ATR stop: stop {stop:.2f}, distance {abs(entry-stop):.2f} ({abs(entry-stop)/entry*100:.2f}%), qty {qty:.0f}")
for mult in (1.0, 1.5, 2.5, 3.0):
    sim_stop_dist = mult * atr
    scaled_qty = qty * (2.0 / mult)  # qty scales as risk/distance → 1/mult
    print(f"  hypothetical {mult}×ATR: distance {sim_stop_dist:.2f} ({sim_stop_dist/entry*100:.2f}%), qty would be {scaled_qty:.0f}")

## 4. Materialize the orders — `obb.techtrade.orders`

The plan carries the orders inline. `obb.techtrade.orders(plan=plan)` is the idempotent re-validation that confirms the plan round-trips cleanly through a JSON boundary (which matters when plans are exported to Excel or sent across a network). Returns the same `list[Order]`.

### Why split one trade into multiple order legs?

A single trade decision — *"buy NVDA at $180, target $195, stop at $172, close by day 20 if neither hits"* — becomes **four separate orders** sitting at the broker:

1. An [**entry order**](https://www.investopedia.com/terms/e/entryorder.asp) that fires once at bar `t+1` open.
2. A [**stop-loss order**](https://www.investopedia.com/terms/s/stop-lossorder.asp) that sits dormant until price trades through the stop.
3. A [**take-profit / limit order**](https://www.investopedia.com/terms/t/take-profitorder.asp) that sits dormant until price reaches the target.
4. A **time-stop order** that fires at a scheduled bar count.

**Why not just one order that "closes at whatever"?** Two reasons professional trading systems (and every retail platform since ~2010) work this way:

1. **Atomicity under network failure.** If Alex's laptop crashes 3 minutes after entering the trade, the stop and target orders are *already at the broker*. He doesn't lose downside protection because his internet dropped. This is the same [**write-ahead**](https://en.wikipedia.org/wiki/Write-ahead_logging) discipline databases use — commit protective structures before the operation they protect.
2. **Race-condition-free exit selection.** If price gaps down to hit the stop and then rebounds to the target in the same day, **which exit wins?** With separate broker-registered orders the answer is deterministic: **first one to trigger wins**, the other is auto-canceled ([OCO / One-Cancels-Other](https://www.investopedia.com/terms/o/oco.asp)). Without them, Alex is watching his screen making a discretionary decision under pressure — the worst possible time to think.

### The `intent` tag

Techtrade attaches an `intent` string (`entry` / `exit_stop` / `exit_target` / `exit_time` / `exit_signal`) to each order so downstream code — broker adapters, Excel export, journaling — can reason about *why* each order exists without having to reverse-engineer the price levels. This is a [**semantic tag**](https://en.wikipedia.org/wiki/Semantic_HTML) borrowed from HTML design: attach meaning, not just data.

### Order types you'll see

| Type | What it means | Investopedia |
|---|---|---|
| `market` | Fill at the next available price (fastest, worst price) | [Market Order](https://www.investopedia.com/terms/m/marketorder.asp) |
| `limit` | Fill only at the specified price or better | [Limit Order](https://www.investopedia.com/terms/l/limitorder.asp) |
| `stop` | Trigger a market order when price hits the stop | [Stop Order](https://www.investopedia.com/terms/s/stoporder.asp) |
| `stop_limit` | Trigger a limit order when price hits the stop | [Stop-Limit Order](https://www.investopedia.com/terms/s/stop-limitorder.asp) |

`tif` = [**Time-In-Force**](https://www.investopedia.com/terms/t/timeinforce.asp) — how long the order stays live: `day` (expires end-of-day), `gtc` (Good-Til-Cancelled), `ioc` (Immediate-or-Cancel), etc.

In [ ]:
orders = obb.techtrade.orders(plan=plan).results
print(f"--- {len(orders)} order leg(s) for {plan.symbol} ---\n")
for o in orders:
    parts = [
        f"intent={o.intent:<12}",
        f"side={o.side:<10}",
        f"qty={o.quantity}",
        f"type={o.order_type}",
    ]
    if o.limit_price is not None:
        parts.append(f"limit=${o.limit_price}")
    if o.stop_price is not None:
        parts.append(f"stop=${o.stop_price}")
    parts.append(f"tif={o.tif}")
    print("  " + " ".join(parts))

### What each leg means

- **`entry`** is the first leg — typically a market order at `t+1` open (no look-ahead). See [Investopedia: Entry Point](https://www.investopedia.com/terms/e/entry-point.asp).
- **`exit_stop`** is the protective [stop-loss](https://www.investopedia.com/terms/s/stop-lossorder.asp). If price trades through the stop, this leg fires and closes the position at-market. Its purpose: **cap the loss at exactly what the sizing formula planned for**.
- **`exit_target`** is the [take-profit](https://www.investopedia.com/terms/t/take-profitorder.asp) limit. Closes the position when price reaches the target. Sized off the `target_r_multiple` — if `r:r=2.0`, the target is 2× the stop distance away from entry.
- **`exit_time`** is the [**time stop**](https://www.investopedia.com/terms/t/time-decay.asp) (default `max_holding_bars=20` business days). If neither stop nor target has fired by then, close at-market. Named after option time decay but applies to stocks too: a trade that hasn't worked in 4 weeks is telling you the setup was wrong.
- **`exit_signal`** (not always present) fires if the composite confluence flips to the opposite side mid-trade. Represents an adaptive close — the original thesis has been invalidated by new evidence.

### The OCO relationship

**[One-Cancels-Other (OCO)](https://www.investopedia.com/terms/o/oco.asp)** is the crucial semantic that makes exit legs safe to leave at the broker: **the first exit to trigger cancels the others**. Without OCO, hitting the stop AND then rebounding to the target could execute both, leaving Alex with a short position when he originally went long. Every reputable retail broker (Interactive Brokers, Alpaca, TD Ameritrade, Charles Schwab) supports OCO natively; the techtrade broker adapter registers the exit legs as an OCO group.

A real broker integration knows which leg is which from the `intent` tag — it's the contract between techtrade and any downstream execution layer.

## 5. Paper-fill the orders — `obb.techtrade.simulate`

`simulate` walks the order legs forward over a bar window and returns the realized fills. The fill model:

- **Bar-`t` signals fill at `t+1` open.** This is the [**no-look-ahead**](https://www.investopedia.com/terms/l/lookaheadbias.asp) discipline. The engine refuses to fill on the bar that produced the signal.
- **[Slippage](https://www.investopedia.com/terms/s/slippage.asp)** is applied per-fill (default convention; see `engine/execution.py`).
- **[Commission](https://www.investopedia.com/terms/c/commission.asp)** is applied per-fill (configurable).
- **The first exit to trigger wins.** If price hits the stop before the target, you get the stop-fill and the target leg is canceled ([OCO semantics](https://www.investopedia.com/terms/o/oco.asp)).

### Why "no look-ahead" is the single most important discipline

[**Look-ahead bias**](https://www.investopedia.com/terms/l/lookaheadbias.asp) is when a backtest silently uses information it wouldn't have had in real-time. The most common form: computing today's signal from today's *close*, then filling the order at today's close. In reality, if you compute the signal at the close, you can't fill until *tomorrow's open* — the market has closed by the time you have the number.

This one-bar difference sounds trivial. It is **catastrophic**. A momentum strategy that looks great with same-bar fills often loses money with next-bar-open fills, because the "edge" was really just the algorithm cheating on when it saw the data. See Marcos López de Prado's [*Advances in Financial Machine Learning*](https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086) (Wiley 2018) Ch.11 for a formal treatment, or Ernie Chan's [*Algorithmic Trading*](https://www.wiley.com/en-us/Algorithmic+Trading%3A+Winning+Strategies+and+Their+Rationale-p-9781118460146) (Wiley 2013) for retail-scaled examples.

**Every "amazing" retail backtest that fails in production has look-ahead bias somewhere.** The techtrade simulator refuses to fill on the signal bar for exactly this reason. When you see `entry_ts > signal_ts` in the fills below, that's the engine enforcing the discipline that separates real backtests from fantasy.

### Why slippage and commission MUST be modeled

A profitable trading strategy on paper often loses money in production. The two most common reasons:

1. **[Slippage](https://www.investopedia.com/terms/s/slippage.asp)** — the difference between the price you *wanted* to fill at and the price you *actually* fill at. Comes from bid-ask spread, liquidity, and market movement between signal and fill. Small on liquid stocks (~1-3 basis points on NVDA); massive on illiquid ones (50+ bps on microcaps).
2. **[Commission](https://www.investopedia.com/terms/c/commission.asp)** — the broker's fee. Zero at IBKR-Lite / Alpaca / Robinhood; ~$0.005/share at IB Pro; more for institutional. Still adds up when you make hundreds of round-trips a year.

**Rule of thumb**: if your paper strategy's [**expectancy per trade**](https://www.investopedia.com/articles/trading/06/riskrewardexpectancy.asp) is less than 10× your round-trip friction, it probably won't survive contact with a live broker. The simulator lets you see this before deploying real money.

Below we fetch a real forward window (the trailing 30 business days for the chosen symbol) and replay the orders over it.

In [ ]:
# Pull a recent OHLCV window — the forward bars the simulator will walk through.
# Wall-clock: 2-5s cached.
from datetime import date, timedelta
end = date.today()
start = end - timedelta(days=45)  # ~30 business days

ohlcv_obj = obb.equity.price.historical(
    symbol=SYMBOL,
    start_date=str(start),
    end_date=str(end),
    provider="fmp_cached",
)
ohlcv_df = ohlcv_obj.to_dataframe()
print(f"Fetched {len(ohlcv_df)} bars for {SYMBOL} from {start} to {end}.")
ohlcv_df.tail(5)

In [ ]:
# techtrade expects bars in a particular shape (per the README §Commands signature for simulate).
# Convert the OHLCV dataframe into the list-of-dict shape simulate consumes.
from decimal import Decimal

def _to_decimal(x):
    return Decimal(str(round(float(x), 4)))

bars = []
for ts, row in ohlcv_df.iterrows():
    bars.append({
        "symbol": SYMBOL,
        "timestamp": ts.isoformat() if hasattr(ts, "isoformat") else str(ts),
        "open":   _to_decimal(row["open"]),
        "high":   _to_decimal(row["high"]),
        "low":    _to_decimal(row["low"]),
        "close":  _to_decimal(row["close"]),
        "volume": _to_decimal(row["volume"]),
    })
print(f"Prepared {len(bars)} bars for simulate.")
print("First bar:", {k: bars[0][k] for k in ('symbol','timestamp','close')})
print("Last bar:",  {k: bars[-1][k] for k in ('symbol','timestamp','close')})

In [ ]:
# Run the paper-broker forward over the bar window.
fills = obb.techtrade.simulate(orders=orders, bars=bars).results
print(f"--- {len(fills)} paper fill(s) ---\n")
for f in fills:
    print(
        f"  ts={f.timestamp}  {f.side:<10} qty={f.quantity}  price=${f.price}  "
        f"slippage=${f.slippage}  commission=${f.commission}"
    )

### What the fills tell Alex

- **An entry fill at `t+1` open** — confirms the [no-look-ahead](https://www.investopedia.com/terms/l/lookaheadbias.asp) discipline (the signal was at bar `t`, fill is at the NEXT bar). Compare the `timestamp` on the fill vs the `as_of` date on the signal — the fill must be strictly later.
- **An exit fill (stop, target, or time)** — this is what *actually happened* on the historical path. The exit's `intent` is preserved in the order it came from; cross-reference by matching `order_ref` to the orders list. Which exit fired first is often surprising and worth studying.
- **[Slippage](https://www.investopedia.com/terms/s/slippage.asp) non-zero** — the engine assumed the price slipped on entry; that's a real-world cost most naive paper backtests ignore. Look at slippage as a % of position — 5 bps on a 3% expected move is 1.7% of your edge gone before commissions.
- **0 fills** — possible: if the entry never triggered (e.g. for a limit entry that never reached the limit) OR the bar window starts AFTER the signal date. This is **realistic, not a bug**. Real markets often produce signals that never fill; the sizing math already accounts for that (unfilled = no risk taken).

### What a professional trader inspects

Beyond just "did I make money on this run," the trained eye looks for:

- **Fill timestamp discipline** — did the entry fill on the bar AFTER the signal bar? Confirms no look-ahead.
- **Exit path plausibility** — did the exit make sense given the intra-window price action? Or does it look like the sim used data it shouldn't have?
- **Slippage magnitude vs stock's typical spread** — the engine's slippage should be roughly `0.5 × typical_bid_ask_spread` for a liquid stock. A wildly larger or smaller number means the slippage model is miscalibrated for this ticker.
- **"Consistency with the plan"** — every fill should correspond to an order the plan generated. Extra fills, missing fills, or fills at wrong intents indicate a bug worth reporting.

This is [**backtest hygiene**](https://en.wikipedia.org/wiki/Backtesting) — the discipline that separates a system worth trading from a spreadsheet worth deleting. See also [Investopedia: Backtesting Pitfalls](https://www.investopedia.com/terms/b/backtesting.asp).

## 6. Realized P&L on the simulated path

If at least one entry + one exit fill exist, Alex can compute what the trade *actually returned* on the historical window. Compare it to what the recommendation *predicted*.

### What "realized" vs "unrealized" means

- **[Realized P&L](https://www.investopedia.com/terms/r/realizedprofit.asp)** is what you get after the exit fills — locked-in, IRS-reportable, actually in your account.
- **[Unrealized P&L](https://www.investopedia.com/terms/u/unrealizedgain.asp)** is what your open position is worth *right now* — paper only, can evaporate before you close.

The simulator below computes realized P&L because it walks the position to its exit. The distinction matters practically: a trader who chases unrealized gains (celebrates every green candle before closing) invariably gives back the gains on the next reversal. The [**mark-to-market**](https://www.investopedia.com/terms/m/marktomarket.asp) discipline is to think in realized terms — a trade isn't a winner until it's closed at a profit.

### Reading the P&L breakdown

The cell below decomposes P&L into three lines:

1. **Gross** — the pure price difference (`exit - entry) × qty`), before any friction.
2. **Slippage** — what the fill model assumes you actually paid vs the theoretical mid.
3. **Commission** — broker fee.

**Gross − slippage − commission = Net.** Net is what actually lands in your account.

Watch the *ratio* between gross and net: on a small-edge trade, net can be a small fraction of gross even at 0% commission (zero-commission brokers still make money on payment-for-order-flow slippage — see the [SEC on PFOF](https://www.sec.gov/investor/alerts/paymentforder-flow.htm) if you want the disclosure detail). This is what "the frictions matter" looks like in dollars, not percentages.

In [ ]:
if not fills:
    print("No fills — no realized P&L to compute. Try a longer bar window.")
else:
    entries = [f for f in fills if f.side in ("buy", "sell_short")]
    exits = [f for f in fills if f.side in ("sell", "buy_to_cover")]
    if not (entries and exits):
        print("Have entry but no exit yet (position still open in the window), or vice versa.")
    else:
        entry_fill = entries[0]
        exit_fill = exits[0]
        qty = float(entry_fill.quantity)
        ep = float(entry_fill.price)
        xp = float(exit_fill.price)
        slippage_total = float(entry_fill.slippage) + float(exit_fill.slippage)
        commission_total = float(entry_fill.commission) + float(exit_fill.commission)
        if entry_fill.side == "buy":
            gross = (xp - ep) * qty
        else:  # sell_short -> buy_to_cover
            gross = (ep - xp) * qty
        net = gross - slippage_total - commission_total
        notional = ep * qty
        print(f"  entry: ${ep:.2f} × {qty:.0f} = notional ${notional:,.2f}")
        print(f"  exit:  ${xp:.2f}  ({exit_fill.timestamp})")
        print(f"  gross P&L:       ${gross:+,.2f}")
        print(f"  - slippage cost: ${slippage_total:.2f}")
        print(f"  - commission:    ${commission_total:.2f}")
        print(f"  = NET P&L:       ${net:+,.2f}  ({net/notional*100:+.2f}% of notional)")

### What this number means

**It does NOT mean the trade was good.** A single sample of one historical path is **not statistical evidence**. In statistical terms, `n=1` — the [**sample size**](https://www.investopedia.com/terms/s/sample.asp) is one. You cannot infer anything about a strategy's edge from one trade, any more than you can infer a coin is fair from one flip.

**The base rate** — Ernie Chan (*Algorithmic Trading*, Wiley 2013) and Marcos López de Prado (*Advances in Financial Machine Learning*, Wiley 2018) both cite the same rough figure: **you need at least 30 trades to have any meaningful signal about a strategy's win rate, and 100+ before drawing confident conclusions**. Below that, you're staring at [**noise**](https://en.wikipedia.org/wiki/Noise_(signal_processing)). See also [Investopedia: Statistical Significance](https://www.investopedia.com/terms/s/statistical-significance.asp).

The whole point of notebook 04 (`validate`) is to do this thousands of times over many resampled folds and compute a [**PBO (Probability of Backtest Overfitting)**](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2308659) + [**DSR (Deflated Sharpe Ratio)**](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551) — turning a single anecdote into a statistic. These are metrics from David Bailey and Marcos López de Prado's [*Pseudo-Mathematics and Financial Charlatanism*](https://www.ams.org/notices/201405/rnoti-p458.pdf) (*Notices of the AMS*, 2014) — mandatory reading if you're serious about telling a real edge from data-mining artifact.

### But the single sample IS useful for two things

1. **[Sanity-check](https://en.wikipedia.org/wiki/Sanity_check) the engine.** If the rec said BUY and the simulate said "hit stop before target", that's a coherent loss. If the simulate produced something nonsensical (entry without an exit, fills at the same bar as signal, fills at prices outside the bar's high-low range), there's a bug — report it.
2. **Feel the cost of [friction](https://www.investopedia.com/terms/f/frictioncost.asp).** [Slippage](https://www.investopedia.com/terms/s/slippage.asp) + [commission](https://www.investopedia.com/terms/c/commission.asp) together can easily eat 20-50% of a small-edge trade. Seeing it in dollars — not as an abstract 3 basis points — makes Alex re-think a 1.5 R:R setup. **This intuition-building is one of the most important reasons paper trading matters** before real capital is deployed.

### The overconfidence trap

Novice traders looking at a single winning simulated trade often conclude the strategy works. **This is the exact cognitive bias — [**confirmation bias**](https://www.investopedia.com/terms/c/confirmation-bias.asp) plus [**hot-hand fallacy**](https://en.wikipedia.org/wiki/Hot-hand_fallacy) — that Nobel laureate Daniel Kahneman documents in [*Thinking, Fast and Slow*](https://en.wikipedia.org/wiki/Thinking,_Fast_and_Slow)** (FSG 2011). The discipline of *always* moving from single-trade evidence to statistical validation (notebook 04) is what a systematic trader adds to their process to counteract the bias.

Elder's rule: **"Never trust one trade. Never distrust the statistics."**

## 7. Variation: same symbol, different preset

Re-run the WHOLE chain (signal → plan → simulate) under the other two presets to see how the trade shape changes. Trend-followers and mean-reverters disagree about this exact ticker on this exact day.

### Why two smart strategies can disagree on the same stock

This is worth internalizing: **the market is not a single game.** [**Trend-following**](https://www.investopedia.com/terms/t/trendtrading.asp) and [**mean-reversion**](https://www.investopedia.com/terms/m/meanreversion.asp) are two literally opposite theories of price behavior, both of which have been empirically supported by decades of academic research and real-money trading:

- **Trend followers** believe today's move predicts tomorrow's move (auto-correlated returns; positive [**momentum**](https://www.investopedia.com/terms/m/momentum.asp)). Empirical support: Asness/Moskowitz/Pedersen [*"Value and Momentum Everywhere"*](https://onlinelibrary.wiley.com/doi/10.1111/jofi.12021) (*Journal of Finance* 2013), plus decades of returns from AQR, Man AHL, and Winton.
- **Mean reverters** believe today's move overshoots and tomorrow's will pull back (negative auto-correlation on short timescales). Empirical support: Poterba & Summers [*"Mean Reversion in Stock Prices"*](https://www.nber.org/papers/w2343) (*JFE* 1988), plus the entire statistical arbitrage industry (Renaissance Technologies, D.E. Shaw, Two Sigma).

**Both can be right** — because they operate on different time scales and different market regimes:

| Preset | Best regime | Worst regime | Time scale |
|---|---|---|---|
| `trend_follow` | Strong directional tape | Choppy sideways | Weeks to months |
| `mean_revert` | Range-bound, high vol | Persistent trends | Hours to days |
| `breakout` | Volatility compression → expansion | Extended chop | Days to weeks |

When they disagree, the market is in a regime where their theories give opposite predictions — which is information about the tape, not a bug in either preset.

### The value of running all three

Two use cases:

1. **Convergence = high-confidence signal.** When all three presets flag BUY on the same name, you have three literally-opposing theoretical frameworks all agreeing — that's stronger than any single-framework conviction.
2. **Divergence = "understand the setup before trading it."** When they disagree, don't just pick the loudest one. Read the [audit trail](https://www.investopedia.com/terms/a/audittrail.asp) (§2 above) under each preset to understand *why* they disagree. Often the answer is *"this is a range-bound day; trend-follow is fighting the tape"* — in which case you should trust `mean_revert`, not average the signals.

In [ ]:
comparison = []
for p_name in ("trend_follow", "mean_revert", "breakout"):
    p_plans = obb.techtrade.plan(symbols=[SYMBOL], preset=p_name, risk=RISK).results
    if not p_plans:
        comparison.append({"preset": p_name, "action": "(no plan)", "score": None, "r:r": None, "qty": None})
        continue
    pp = p_plans[0]
    comparison.append({
        "preset": p_name,
        "score": round(pp.signal.score, 4),
        "action": pp.recommendation.action,
        "conviction": pp.recommendation.conviction,
        "entry": float(pp.recommendation.entry_price),
        "stop": float(pp.recommendation.stop_price),
        "target": float(pp.recommendation.target_price),
        "r:r": round(pp.recommendation.risk_reward, 2),
        "qty": float(pp.recommendation.position_size),
    })

pd.DataFrame(comparison)

### What disagreement looks like

- **All three agree on direction** (all BUY or all SELL_SHORT) and roughly on entry/stop/target — **strong signal**. The presets emphasize different facets but the underlying setup is robust. This is the "convergent evidence" case worth taking a full-size position on (subject to the checklist in §8).
- **Two agree, one is FLAT** — the dissenting preset finds insufficient signal under its weighting. Read the dissenter's votes (run cell 2 again with that preset) to see WHICH family is dragging the composite down. **Halve position size** as a general rule when only 2/3 presets agree.
- **Split signals (e.g. trend_follow says BUY, mean_revert says SELL_SHORT)** — Alex usually stays out. He's looking at a name where the bull/bear case is balanced; neither side has an edge. The [**expected value**](https://www.investopedia.com/terms/e/expected-return.asp) of taking either side under this evidence is approximately zero after commissions.

### The "regime detection" hint

If `mean_revert` and `breakout` both flag your ticker but `trend_follow` doesn't, you're likely looking at a name in a **choppy consolidation range** — the ticker is bouncing between support and resistance without a clear trend. Trend followers correctly refuse to trade this; mean-reverters and breakout traders correctly identify the setup for their respective plays. This kind of "which preset agrees" pattern is a **cheap [regime classifier](https://www.investopedia.com/terms/m/market_cycles.asp)** — see notebook 04's validation gate for how techtrade formalizes regime awareness into an anti-overfit filter.

## 8. The single-position checklist

Before Alex takes a real position on the symbol he studied here:

- [ ] **`score` is decisively past the entry threshold** (not just barely; he wants `|score| >= 0.5` for personal margin). Marginal signals are indistinguishable from noise on a small sample; wait for stronger evidence.
- [ ] **[`r:r`](https://www.investopedia.com/terms/r/riskrewardratio.asp) >= 2.0** so the math works even at his historical hit rate (~45%). Below 2.0 requires a >50% win rate; retail traders almost never achieve that consistently.
- [ ] **The audit trail (votes) shows family agreement** — not just one indicator at full weight. See §2 interpretation.
- [ ] **The simulated forward window's exit makes sense** — stop hits before target on losers, target before stop on winners, no fills-on-same-bar anomalies.
- [ ] **The trade has been through `validate`** (notebook 04). A single backtest is anecdote; a [**PBO**](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2308659) + [**DSR**](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551) + verdict is statistical evidence.

If any of those fail, **skip the trade** or re-run with a different preset / different ticker.

### The professional's rule

Alexander Elder (*[Trading for a Living](https://en.wikipedia.org/wiki/Alexander_Elder)*, Wiley 1993): *"The amateur trader looks for a reason to trade; the professional looks for a reason not to."*

Every item in the checklist above is a **veto**, not a **confirmation**. The default state is "don't trade." A trade only happens when every veto is cleared. This is the [**default-deny security model**](https://en.wikipedia.org/wiki/Default_deny) applied to trading: unknown = don't do it, known = maybe do it. The alternative (default-allow: trade unless you spot a red flag) is what discretionary traders do, and it's why 80-90% of them lose money over any five-year window ([FINRA on retail trading outcomes](https://www.finra.org/investors/insights/day-trading-real-cost)).

---

## Recommended reading — go deeper

**Books directly relevant to what this notebook does:**

- Van K. Tharp, *[Trade Your Way to Financial Freedom](https://vantharp.com/trade-your-way-to-financial-freedom/)* (McGraw-Hill, 1998) — expectancy, position sizing, R-multiples. The single most useful trading book for a retail systematic trader.
- Alexander Elder, *[Trading for a Living](https://en.wikipedia.org/wiki/Alexander_Elder)* (Wiley, 1993) — psychology, discipline, the [**Triple Screen**](https://www.investopedia.com/terms/t/tripletop.asp) method.
- Ernie Chan, *[Algorithmic Trading](https://www.wiley.com/en-us/Algorithmic+Trading%3A+Winning+Strategies+and+Their+Rationale-p-9781118460146)* (Wiley, 2013) — the applied bridge from quant theory to real-money retail systems; excellent on execution details.
- Marcos López de Prado, *[Advances in Financial Machine Learning](https://www.wiley.com/en-us/Advances+in+Financial+Machine+Learning-p-9781119482086)* (Wiley, 2018) — the modern reference for avoiding look-ahead bias, backtest overfitting, and statistical significance in trading strategy research.
- Jack Schwager, *[Market Wizards](https://en.wikipedia.org/wiki/Market_Wizards)* (NYIF, 1989) — interviews with the traders who made 100×+ returns; recurring themes: **cut losses fast, let winners run, size positions rigidly**.

**Free / academic references linked above:**

- Bailey & López de Prado, [*Pseudo-Mathematics and Financial Charlatanism*](https://www.ams.org/notices/201405/rnoti-p458.pdf) (*Notices of the AMS*, 2014).
- Asness, Moskowitz & Pedersen, [*"Value and Momentum Everywhere"*](https://onlinelibrary.wiley.com/doi/10.1111/jofi.12021) (*Journal of Finance*, 2013).
- [SEC investor.gov](https://www.investor.gov/) — regulator-published beginner education.
- [FINRA Investor Education](https://www.finra.org/investors/learn-to-invest) — includes the sobering day-trading outcomes data.

## What's next in the series

- **Notebook 04 — The Validation Gate**: take this exact plan and call `obb.techtrade.validate(plan, method="wfo", horizon_years=5)`. [Walk-forward folds](https://www.investopedia.com/terms/w/walkforward.asp) + [PBO](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2308659) + [DSR](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=2460551) + verdict. The anti-[overfit](https://www.investopedia.com/terms/o/overfitting.asp) gate that turns the single simulated path above into statistical evidence.
- **Notebook 05 — Per-Sector Tuning**: would different indicator periods have given a different score on this ticker? `obb.techtrade.tune(segment=...)` proposes new periods per-sector; persists only the ones that pass `validate`.
- **Notebook 06 — Audit and Replay**: cross-reference what Alex actually did with what the engine suggested. Journal entry. End-of-day discipline. See [Investopedia on trading journals](https://www.investopedia.com/articles/trading/09/trading-journal.asp) — the single highest-leverage skill-building habit that retail traders almost never adopt.

---

*End of notebook 03. Series: "A Developer Guide to Disciplined Trading". Maintained on the `trading_technicals` branch of `prajoria/OpenBB`.*